# Day 15 — Customer growth, churn and plan-wise revenue trends

Customer definitions follow the month-end paid subscription snapshot: non-trial, positive MRR, start_date <= month_end and (end_date is null or end_date >= month_end). End-date inclusivity remains provisional.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path.cwd(); DATA=ROOT/'data/processed/week3'
customer_trends=pd.read_csv(DATA/'customer_growth_churn_monthly.csv',parse_dates=['report_month'])
plan_trends=pd.read_csv(DATA/'plan_wise_monthly_trends.csv',parse_dates=['report_month'])
customer_trends.head()

## Customer growth and churn

In [ ]:
ax=customer_trends.plot(x='report_month',y=['active_paid_customers','new_paid_customers','reactivated_paid_customers','churned_paid_customers'],marker='o',title='Active paid customers, new customers and churn')
ax.set_ylabel('Distinct accounts'); plt.tight_layout(); plt.show()

The first observed month has no prior month-end snapshot, so new/churn counts are NULL. Subsequent new and churn counts compare distinct account sets between adjacent month-end snapshots. The final month is not treated as churn beyond the observation window.

## Plan-wise MRR and active customers

In [ ]:
pivot=plan_trends.pivot(index='report_month',columns='plan_tier',values='mrr_source_amount').fillna(0)
ax=pivot.plot(marker='o',title='Plan-wise month-end MRR (source amount units)')
ax.set_ylabel('MRR source amount units'); plt.tight_layout(); plt.show()

In [ ]:
plan_trends.pivot(index='report_month',columns='plan_tier',values='active_paid_customers').plot(marker='o',title='Distinct active accounts by subscription plan')
plt.ylabel('Distinct accounts within plan'); plt.tight_layout(); plt.show()

Plan-level active customer counts are distinct within each plan and can double-count an account across plans if it holds concurrent subscriptions. Plan MRR sums eligible subscription MRR. Plan mix changes should not be interpreted as total customer growth.

## Retention and revenue interpretation

Use `cohort_retention_matrix.csv` for cohort retention and `revenue_retention_summary.csv` for opening-base GRR/NRR. GRR caps each opening customer contribution at their opening MRR; NRR includes expansion/contraction among the same opening customer base and excludes new/reactivated customers.